## 1. Configure your environment
pip install -r requirements.txt

## 2. Insure your data directory as following

```text
SpaAGMF/
├── config/
├── data/
├── data_process/
├── dataset/
├── models/
├── pretrained_model_uni/
├── adan.py
├── cross_platform&batch_main.py
├── cross_sample_main.py
├── train.py
├── tutorial.ipynb
└── utils.py
```

## 3. Running example

In [ ]:
import os
import hydra
from datetime import datetime
from hydra.core.global_hydra import GlobalHydra

# 1. 导入两个不同的 main 函数，并使用 as 设置别名以防止命名冲突
from cross_sample_main import main as cross_sample_main_func
from cross_platform_main import main as cross_platform_main_func

def run_experiment(target_dataset: str, main_func: callable):
    """
    Helper function to set up Hydra and run the specified main function.
    """
    # Clear any existing Hydra instances
    GlobalHydra.instance().clear()

    # Initialize Hydra
    hydra.initialize(version_base="1.1", config_path="./config")

    # Compose the configuration
    cfg = hydra.compose(
        config_name="config",
        overrides=[
            f"dataset={target_dataset}",
            f"ctt={target_dataset}",
            f"cls={target_dataset}",
            f"augment={target_dataset}",
            f"dataset_name={target_dataset}"
        ]
    )

    # Extract dataset name and format time
    dataset_name = cfg.dataset_name
    now = datetime.now()
    date_str = now.strftime('%Y-%m-%d')
    time_str = now.strftime('%H-%M-%S')

    # Construct run directory path
    run_dir = os.path.join("outputs", dataset_name, date_str, time_str)

    # Get current working directory and reset to root if inside "outputs"
    base_dir = os.getcwd()
    if "outputs" in base_dir:
        base_dir = base_dir.split("outputs")[0]
        os.chdir(base_dir)

    # Create and change to the new run directory
    full_run_dir = os.path.join(base_dir, run_dir)
    os.makedirs(full_run_dir, exist_ok=True)
    os.chdir(full_run_dir)

    print(f"\n[{datetime.now()}] Started running {target_dataset}...")
    print(f"Working directory set to: {full_run_dir}\n")

    # Execute the passed main function
    main_func(cfg)

    # IMPORTANT: Change back to the base directory after execution
    # so the next experiment starts from the correct root path.
    os.chdir(base_dir)

In [ ]:
# Perform cross-sample validation (CRC, STHBC)
run_experiment(target_dataset="CRC", main_func=cross_sample_main_func)

In [ ]:
# Perform cross-platform&batch validation (ViHBC, XeHBC, IDC)
run_experiment(target_dataset="ViHBC", main_func=cross_platform_main_func)